In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os
os.chdir("/home/j/jackbarker/surface_predictor")
import pickle
import pandas as pd

import scripts.analyse_eofs as analyse_eofs
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def preprocess(ds):
    # Remove batch dimension if it exists
    if 'batch' in ds.dims:
        ds = ds.squeeze(dim="batch")
    return ds

def preprocess_era5_dataset(ds):
    ds = ds.rename({"valid_time": "datetime", "latitude": "lat", "longitude": "lon"})
    ds = ds.set_index({"datetime": "datetime"})
    ds = ds.reindex(lat=ds.lat[::-1])  # Reverse latitude order to match Graphcast data
    ds = ds.drop_vars(["number", "expver"])
    return ds

def average_area(start_date, end_date, tod, da):
    da = da.sel(lat=slice(2,16), lon=slice(32,45)).sel(datetime=slice(start_date, end_date))
    da = da.sel(datetime=da.datetime.dt.hour == tod)
    return da.mean(dim=["lat", "lon"])

def custom_average_area(start_date, end_date, tod, da, latslice, lonslice):
    da = da.sel(lat=latslice, lon=lonslice).sel(datetime=slice(start_date, end_date))
    da = da.sel(datetime=da.datetime.dt.hour == tod)
    return da.mean(dim=["lat", "lon"])

# Statistics

In [ ]:
error_data_3y = xr.open_mfdataset(
    [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/errors_{day.strftime('%Y%m%d')}-06_n0.nc" for day in pd.date_range("2014-01-01", "2016-12-31")],
    concat_dim="datetime",
    combine="nested",
    preprocess=preprocess,
)

In [ ]:
error_ts_3y = average_area("2014-01-01", "2016-12-31", 6, error_data_3y["2m_temperature"]).load()

In [ ]:
error_dates_3y = error_ts_3y.datetime[error_ts_3y < -1].drop_vars("batch")
good_dates_3y = error_ts_3y.datetime[error_ts_3y >= -1].drop_vars("batch")

In [ ]:
ds = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/86bd6ea11427e91cff390f7f49a40f16/data_stream-oper_stepType-accum.nc"))
ds

In [ ]:
data_2016 = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/2016_land.nc"))
data_2015 = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/2015_land.nc"))
data_2014 = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/2014_land.nc"))

tcc_data = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/2014_2016_tcc.nc"))
precip_data = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/2014_2016_precip.nc"))

# get long names of each variable
long_names = {}
units = {}
for ds in [data_2014, data_2015, data_2016, tcc_data, precip_data]:
    for var in ds.data_vars:
        long_names[var] = ds[var].attrs.get("long_name", "")
        units[var] = ds[var].attrs.get("units", "")

In [ ]:
surface_data = xr.concat([data_2014, data_2015, data_2016], dim="datetime").sel(lat = slice(2, 16), lon=slice(32, 45)).mean(dim=["lat", "lon"])
atm_data = xr.merge([tcc_data, precip_data]).sel(lat = slice(2, 16), lon=slice(32, 45)).mean(dim=["lat", "lon"])

In [ ]:
surface_data

In [ ]:
data = xr.merge([surface_data, atm_data])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax2 = ax.twinx()

data.sel(datetime=good_dates_3y)["src"].plot.hist(ax=ax, alpha=0.5, label="Good Dates")
data.sel(datetime=error_dates_3y)["src"].plot.hist(ax=ax2, alpha=0.5, label="Error Dates", color="red")
ax.legend()

In [ ]:
# Twin-axis histogram with shared bins for good vs error dates
import numpy as np
import matplotlib.pyplot as plt

# Extract 1D arrays for the selected variable after spatial averaging
da_good = data.sel(datetime=good_dates_3y)["src"]
da_err = data.sel(datetime=error_dates_3y)["src"]

v_good = np.asarray(da_good.values).ravel()
v_err = np.asarray(da_err.values).ravel()
v_good = v_good[np.isfinite(v_good)]
v_err = v_err[np.isfinite(v_err)]

# Compute common bin edges across both sets
if v_good.size and v_err.size:
    combined = np.concatenate([v_good, v_err])
elif v_good.size:
    combined = v_good
else:
    combined = v_err
edges = np.histogram_bin_edges(combined, bins=40)

fig, ax = plt.subplots(figsize=(10, 6))
ax2 = ax.twinx()

# Plot with identical bins
ax.hist(v_good, bins=edges, alpha=0.5, label="Good Dates", color="C0")
ax2.hist(v_err, bins=edges, alpha=0.5, label="Error Dates", color="C3")

ax.set_xlabel("src")
ax.set_ylabel("Good count", color="C0")
ax2.set_ylabel("Error count", color="C3")
ax.set_title("Distribution of 'src' for Good vs Error Dates (shared bins)")

# Combined legend
l1, lab1 = ax.get_legend_handles_labels()
l2, lab2 = ax2.get_legend_handles_labels()
ax.legend(l1 + l2, lab1 + lab2, loc="best")

plt.show()

In [ ]:
# Multi-panel twin-axis histograms per variable (shared bins in each subplot)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


# Variables in 'data' after spatial averaging
var_names = [vn for vn in list(data.data_vars) if np.issubdtype(data[vn].dtype, np.number)]
nvars = len(var_names)
if nvars == 0:
    raise ValueError("No numeric variables found in 'data'.")

# Grid layout
ncols = 5 if nvars >= 5 else nvars
nrows = int(np.ceil(nvars / ncols))
fig, axs = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.6*nrows), squeeze=False)

for i, vn in enumerate(var_names):
    r, c = divmod(i, ncols)
    ax = axs[r][c]
    ax2 = ax.twinx()

    da_good = data[vn].sel(datetime=good_dates_3y)
    da_err = data[vn].sel(datetime=error_dates_3y)
    v_good = np.asarray(da_good.values).ravel()
    v_err = np.asarray(da_err.values).ravel()
    v_good = v_good[np.isfinite(v_good)]
    v_err = v_err[np.isfinite(v_err)]

    if v_good.size == 0 and v_err.size == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
        continue

    combined = v_good if v_err.size == 0 else (v_err if v_good.size == 0 else np.concatenate([v_good, v_err]))
    edges = np.histogram_bin_edges(combined, bins=40)

    # good = np.histogram(v_good, bins=edges)[0]
    # error = np.histogram(v_err, bins=edges)[0]

    # centres = (edges[:-1] + edges[1:]) / 2

    # ax.plot(centres, good, label="Count on normal days", color="C0")
    # ax2.plot(centres, error, label="Count on days where large T2m error is seen", color="C3")

    # # shade sqrt(n) poisson counting uncertainties
    # ax.fill_between(centres, good-np.sqrt(good), good+np.sqrt(good), color='C0', alpha=0.2)
    # ax2.fill_between(centres, error-np.sqrt(error), error+np.sqrt(error), color='C3', alpha=0.2)

    ax.hist(v_good, bins=edges, alpha=0.5, label='Count on normal days', color='C0')
    ax2.hist(v_err, bins=edges, alpha=0.5, label='Count on days where large T2m error is seen', color='C3')

    ax.set_title(long_names[vn])
    ax.set_xlabel(f"{long_names[vn]} ({units[vn]})")
    ax.set_ylabel('Normal days', color='C0')
    ax2.set_ylabel('Large T2m error days', color='C3')

# Hide any unused subplots
for j in range(nvars, nrows*ncols):
    r, c = divmod(j, ncols)
    axs[r][c].set_visible(False)

# Single figure legend and title
handles = [Patch(facecolor='C0', alpha=0.5, label='Count on normal days'),
           Patch(facecolor='C3', alpha=0.5, label='Count on days where large T2m error is seen')]
fig.legend(handles=handles, loc='upper right', bbox_to_anchor=(1,1.05), frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("summer_figs/africa_histograms.pdf")

# Timeseries at 4,5,6,7AM

In [ ]:
more_2mt_times = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/localised_2mt_2016_more_times.nc"))

times = average_area("2016-01-01", "2016-12-31", 6, more_2mt_times["t2m"]).datetime


In [ ]:
fig, axs = plt.subplots(5, 1, figsize=(12, 20), sharex=True)

for i, ax in enumerate(axs):
    axs[i].plot(times, average_area("2016-01-01", "2016-12-31", i+4, more_2mt_times["t2m"]), label=f"ERA5 2m Temperature at {i+4}am")
    axs[i].set_title(f"ERA5 2m Temperature at {i+4}am")
    axs[i].set_ylabel("Temperature (K)")

    for error_date in error_dates:
        axs[i].axvline(error_date.to_numpy(), color='red', linestyle='--', alpha=0.5)



# IFS and ERA5/GC comparison

In [ ]:
def preprocess_ifs_data(ds):
    ds = ds.rename({
        'latitude': 'lat',
        'longitude': 'lon',
        'time': 'datetime',
    })
    ds = ds.reindex(lat = ds.lat[::-1])
    return ds

ifs_analysis = preprocess_ifs_data(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/ifs_data/analysis2016.nc"))
ifs_forecast = preprocess_ifs_data(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/ifs_data/da_forecasts2016.nc"))
era5_data = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/localised_2mt_2016.nc"))
gc_forecast = xr.open_mfdataset(
    [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/pred_2016{day.strftime('%m%d')}-06_n0.nc" for day in pd.date_range("2016-01-01", "2016-12-31")],
    concat_dim="datetime",
    combine="nested",
    preprocess=preprocess,
).rename({"time": "datetime"}).set_index({"datetime": "datetime"})


In [ ]:
ifs_forecasts_series = average_area("2016-01-01", "2016-07-01", 6, ifs_forecast["t2m"])
ifs_analysis_series = average_area("2016-01-01", "2016-07-01", 6, ifs_analysis["t2m"])
era5_series = average_area("2016-01-01", "2016-07-01", 6, era5_data["t2m"])
gc_series = average_area("2016-01-01", "2016-07-01", 6, gc_forecast["2m_temperature"]).load()

shared_dates = np.intersect1d(np.intersect1d(ifs_forecasts_series.datetime.values, ifs_analysis_series.datetime.values), np.intersect1d(era5_series.datetime.values, gc_series.datetime.values))


ifs_forecasts_series = ifs_forecasts_series.sel(datetime=shared_dates)
ifs_analysis_series = ifs_analysis_series.sel(datetime=shared_dates)
era5_series = era5_series.sel(datetime=shared_dates)
gc_series = gc_series.sel(datetime=shared_dates)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

axs[0].plot(era5_series.datetime, era5_series, label="ERA5 Analysis", color="blue")
axs[0].set_title("ERA5 and IFS Analysis")
axs[0].set_ylabel("Temperature (K)")
axs[0].plot(ifs_analysis_series.datetime, ifs_analysis_series, label="4D-Var Operational IFS Analysis", color="green")
axs[0].legend()

axs[1].plot(gc_series.datetime, gc_series, label="Graphcast 2m Temperature 6h Forecast", color="orange")
axs[1].set_title("GraphCast and IFS Forecasts")
axs[1].set_ylabel("Temperature (K)")
axs[1].plot(ifs_forecasts_series.datetime, ifs_forecasts_series, label="4D-Var Operational 12hr IFS Forecasts", color="purple")
axs[1].legend()    


strange_days = era5_series.datetime[np.abs(era5_series-gc_series) > 1]

for i in range(2):
    # plot vertical lines at each strange day
    for strange_day in strange_days:
        axs[i].axvline(strange_day.to_numpy(), color='red', linestyle='--', alpha=0.5)

plt.savefig("summer_figs/ifs_comparison.pdf")